In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torchinfo import summary
from tqdm import tqdm
from torchvision.datasets import MNIST

#==设置随机种子
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
device

device(type='cuda')

In [3]:
torch.backends.cudnn.benchmark=True

In [4]:
    train_dataset = MNIST(root='mnist_data/', train=True,
                     download=True, transform=transforms.ToTensor())

In [5]:
    test_dataset = MNIST(root='mnist_data/', train=False,
                     download=True, transform=transforms.ToTensor())

In [6]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=(device.type == 'cuda'),num_workers=0,)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True,pin_memory=(device.type == 'cuda'),num_workers=0,)

In [7]:
class TeacherModel(nn.Module):
    def __init__(self,in_channels=1,num_classes=10):
        super(TeacherModel,self).__init__()
        self.relu=nn.ReLU()
        self.fc1=nn.Linear(784,1200)
        self.fc2=nn.Linear(1200,1200)
        self.fc3=nn.Linear(1200,num_classes)
        self.dropout= nn.Dropout(p=0.5)
    def forward(self,x):
        x=x.view(-1,784)
        x=self.fc1(x)
        x=self.dropout(x)
        x=self.relu(x)
        
        x=self.fc2(x)
        x=self.dropout(x)
        x=self.relu(x)
        
        x=self.fc3(x)
        return x

In [8]:
model=TeacherModel()
model=model.to(device)

In [9]:
summary(model)

Layer (type:depth-idx)                   Param #
TeacherModel                             --
├─ReLU: 1-1                              --
├─Linear: 1-2                            942,000
├─Linear: 1-3                            1,441,200
├─Linear: 1-4                            12,010
├─Dropout: 1-5                           --
Total params: 2,395,210
Trainable params: 2,395,210
Non-trainable params: 0

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=1e-4)

In [12]:
teacher_model=model

In [13]:
class StudentModel(nn.Module):
    def __init__(self,in_channels=1,num_classes=10):
        super(StudentModel,self).__init__()
        self.relu=nn.ReLU()
        self.fc1=nn.Linear(784,20)
        self.fc2=nn.Linear(20,20)
        self.fc3=nn.Linear(20,num_classes)
        self.dropout= nn.Dropout(p=0.5)
    def forward(self,x):
        x=x.view(-1,784)
        x=self.fc1(x)
        #x=self.dropout(x)
        x=self.relu(x)
        
        x=self.fc2(x)
        #x=self.dropout(x)
        x=self.relu(x)
        
        x=self.fc3(x)
        return x


In [14]:

model=StudentModel()
model=model.to(device)


In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=1e-4)



In [17]:
#准备预训练好的教师模型
teacher_model.eval()

#准备新的学生模型
model=StudentModel()
model=model.to(device)
model.train()

temp= 7

In [ ]:
#hard_loss
hard_loss=nn.CrossEntropyLoss()
#hard_loss 权重
alpha= 0.3

#soft_loss 
soft_loss=nn.KLDivLoss(reduction="batchmean")
optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)